# Analise de Acoes e Fundamentos

Analise de acoes brasileiras: precos, dados fundamentalistas, dividendos e setores.

Fontes:
- **YahooFinanceFetcher**: precos OHLCV, fundamentos (P/L, P/VP, ROE, DY), dividendos
- **CVMFetcher**: demonstracoes financeiras oficiais (DFP, ITR)
- **DDMFetcher**: cotacoes, indicadores e FIIs (requer API key)
- **MarketSectorAnalyzer**: performance setorial
- **EconomicSectorAnalyzer**: setores da economia real (IBGE)

**Nota**: Yahoo Finance nao requer API key. CVM e livre mas endpoints podem estar indisponiveis.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

# Tickers de exemplo
TICKERS = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "WEGE3.SA"]

## 1. Precos Historicos — Yahoo Finance

In [ ]:
from carteira_auto.data.fetchers import YahooFinanceFetcher

yahoo = YahooFinanceFetcher()

# Precos atuais
precos = yahoo.get_multiple_prices(TICKERS)
print("Precos atuais:")
for ticker, preco in precos.items():
    print(f"  {ticker}: R${preco:.2f}" if preco else f"  {ticker}: indisponivel")

In [ ]:
# Historico 1 ano
historico = yahoo.get_historical_price_data(TICKERS, period="1y")
print(f"Shape: {historico.shape}")

# Plotar retorno acumulado normalizado
fig, ax = plt.subplots()
for ticker in TICKERS:
    try:
        if isinstance(historico.columns, pd.MultiIndex):
            close = historico[(ticker, "Close")].dropna()
        else:
            close = historico["Close"].dropna()
        retorno_norm = close / close.iloc[0] * 100
        ax.plot(retorno_norm.index, retorno_norm.values, label=ticker.replace(".SA", ""))
    except (KeyError, TypeError):
        pass

ax.axhline(y=100, color="gray", linestyle="--", alpha=0.5)
ax.set_title("Retorno Normalizado (base 100) — Ultimo Ano")
ax.set_ylabel("Base 100")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Dados Fundamentalistas

In [ ]:
# Multiplos fundamentalistas via get_batch_info
campos = [
    "trailingPE", "priceToBook", "dividendYield",
    "returnOnEquity", "debtToEquity", "marketCap"
]
info = yahoo.get_batch_info(TICKERS, fields=campos)

print("Multiplos Fundamentalistas:")
print(info.to_string())

In [ ]:
# Info detalhada de um ativo
petr4_info = yahoo.get_basic_info("PETR4.SA")
print("PETR4 — Informacoes basicas:")
for key in ["shortName", "sector", "industry", "marketCap", "trailingPE", "dividendYield"]:
    val = petr4_info.get(key, "N/A")
    print(f"  {key}: {val}")

## 3. Dividendos

In [ ]:
# Historico de dividendos
divs = yahoo.get_dividends("PETR4.SA")
print(f"PETR4 dividendos: {len(divs)} pagamentos")
if not divs.empty:
    print(divs.tail(10))

    # Total ultimos 12 meses
    from datetime import datetime, timedelta
    cutoff = datetime.now() - timedelta(days=365)
    divs_12m = divs[divs.index >= cutoff]
    print(f"\nTotal dividendos 12m: R${divs_12m.sum().iloc[0]:.2f}" if not divs_12m.empty else "")

## 4. CVM — Demonstracoes Financeiras Oficiais

In [ ]:
from carteira_auto.data.fetchers import CVMFetcher

cvm = CVMFetcher()

# Cadastro de empresas
cadastro = cvm.get_company_registry()
print(f"Empresas na CVM: {len(cadastro)}")

# Mapeamento ticker -> CNPJ (util para buscar DFP por ticker)
try:
    mapa = cvm.build_ticker_cnpj_map()
    print(f"Mapa ticker->CNPJ: {len(mapa)} tickers")
    for t in ["PETR4", "VALE3", "ITUB4"]:
        cnpj = mapa.get(t, "nao encontrado")
        print(f"  {t}: {cnpj}")
except Exception as e:
    print(f"Erro ao construir mapa: {e}")

In [ ]:
# DFP — Demonstracao do Resultado (DRE)
try:
    dre = cvm.get_dfp_by_ticker("PETR4", year=2023, statement="DRE")
    print(f"DRE Petrobras 2023: {len(dre)} linhas")
    print(dre.head(10))
except Exception as e:
    print(f"CVM DFP indisponivel: {e}")
    print("Nota: endpoint CVM pode estar temporariamente fora do ar")

## 5. DDM — Dados de Mercado (API paga)

In [ ]:
DDM_OK = bool(os.getenv("DADOS_MERCADO_API_KEY"))
print(f"DDM API key configurada: {DDM_OK}")

if DDM_OK:
    from carteira_auto.data.fetchers import DDMFetcher
    ddm = DDMFetcher()

    # Cotacoes
    cotacoes = ddm.get_quotations("PETR4", period_init="2024-01-01")
    print(f"Cotacoes PETR4: {len(cotacoes)} dias")

    # Indicadores de mercado
    indices = ddm.get_market_indices()
    print(f"Indices: {len(indices)} registros")

    # Lista de FIIs
    fiis = ddm.get_fii_list()
    print(f"FIIs listados: {len(fiis)}")

## 6. Analise Setorial

In [ ]:
from carteira_auto.analyzers import MarketSectorAnalyzer, EconomicSectorAnalyzer
from carteira_auto.core.engine import PipelineContext

# Setores de mercado (Yahoo Finance)
msa = MarketSectorAnalyzer()
ctx = PipelineContext()
ctx = msa.run(ctx)

if "market_sectors" in ctx:
    print("Setores de mercado:")
    for sector in ctx["market_sectors"]:
        print(f"  {sector.name}: {sector.value}")

if ctx.has_errors:
    print(f"\nErros parciais: {ctx.errors}")

In [ ]:
# Setores da economia real (IBGE)
esa = EconomicSectorAnalyzer()
ctx2 = PipelineContext()
ctx2 = esa.run(ctx2)

if "economic_sectors" in ctx2:
    print("Setores economicos (IBGE):")
    for sector in ctx2["economic_sectors"]:
        print(f"  {sector.name}: {sector.value}")

if ctx2.has_errors:
    print(f"\nErros parciais: {ctx2.errors}")

## Resumo

| Fonte | Dados | Uso |
|-------|-------|-----|
| Yahoo Finance | Precos, fundamentos, dividendos | Analise quantitativa diaria |
| CVM | DFP, DRE oficiais | Validacao de fundamentos |
| DDM | Cotacoes, indicadores, FIIs | Complemento com dados nacionais |
| MarketSectorAnalyzer | Performance setorial | Alocacao setorial |
| EconomicSectorAnalyzer | Crescimento real (IBGE) | Contexto macrosetorial |